In [1]:
ALADIN_KEY = "ttbseoll770145001"
PINECONE_KEY = "pcsk_1JaXF_NB96Zvye9cmG7jnfHAsQy2odgSZ6fefruZ3bcQHMBNbWWQCk3zCbW6niNC2AEfH"

In [2]:
import requests
from typing import List, Dict, Optional

class AladinBookFetchService:
    BASE_URL = "https://www.aladin.co.kr/ttb/api/ItemList.aspx"
    LOOKUP_URL = "https://www.aladin.co.kr/ttb/api/ItemLookUp.aspx"

    def __init__(self, ttb_key: str):
        self.ttb_key = ttb_key

    """
    알라딘 API를 호출해 인기책 / 베스트셀러 리스트를 가져옴
    
    :param query_type: API 호출 타입 (Bestseller, ItemNewAll, ItemNewSpecial, BlogBest 등)
    :param max_results: 가져올 책 개수 (1~100)
    :param search_target: 검색 대상 (Book, Music, DVD 등)
    :param output: 응답 포맷 (js, xml 등)
    :param version: API 버전
    :return: 책 정보 리스트
    """
    def fetch_books(
        self,
        query_type: str = "Bestseller",
        max_results: int = 10,
        search_target: str = "Book",
        output: str = "js",
        version: str = "20131101"
    ) -> Optional[List[Dict]]:
        params = {
            "ttbkey": self.ttb_key,
            "QueryType": query_type,
            "MaxResults": max_results,
            "SearchTarget": search_target,
            "output": output,
            "Version": version,
        }

        try:
            response = requests.get(self.BASE_URL, params=params)
            response.raise_for_status()
            data = response.json()

            items = data.get("item", [])
            books = []
            
            for item in items:
                book = { # 기본 정보
                    "title": item.get("title"),
                    "author": item.get("author"),
                    "publisher": item.get("publisher"),
                    "cover": item.get("cover"),
                    "isbn13": item.get("isbn13"),
                }
                
                isbn13 = item.get("isbn13") # 상세 정보 추가
                if isbn13:
                    detail = self.fetch_book_detail(isbn13)
                    if detail:
                        book.update(detail)  # description, category 추가
                
                books.append(book)
                
            return books

        except requests.RequestException as e:
            print(f"❌ API 호출 오류: {e}")
            return None

    """
    알라딘 API를 통해 특정 도서의 상세 정보를 가져옴
    """
    def fetch_book_detail(self, isbn13: str):
        params = {
            "ttbkey": self.ttb_key,
            "itemIdType": "ISBN13",
            "ItemId": isbn13,
            "output": "js",
            "Version": "20131101",
            "Cover": "Big",
            "OptResult": "cateList,fulldescription,authors"
        }

        try:
            response = requests.get(self.LOOKUP_URL, params=params)
            response.raise_for_status()
            data = response.json()
    
            if "item" not in data or not data["item"]:
                return None
    
            item = data["item"][0]
            
            category_text = item.get("categoryName")
            category_parts = [c.strip() for c in category_text.split(">")][1:]

            return {
                "description": item.get("description", ""),
                "category": category_parts
            }
        
        except requests.RequestException as e:
            print(f"❌ 상세 조회 오류 ({isbn13}): {e}")
            return None

In [3]:
from transformers import AutoTokenizer, AutoModel
import torch

class E5Embedding:
    def __init__(self):
        """
        HuggingFace 모델이랑 토크나이저 같은거 다 불러오자
        참고로 tokenizer은.. 문자를 모델의 입력으로 바꿔주는 도구래! 
        미리 학습되어있는거 불러오면 좋겠지?
        """
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.tokenizer = AutoTokenizer.from_pretrained("intfloat/e5-small") # model_name 넣어주면 알아서 불러와줌
        self.model = AutoModel.from_pretrained("intfloat/e5-small")

    # 입력 텍스트 생성 (타이틀 + 설명 + 저자 등 결합)
    def build_text(self, row):
        parts = [
            f"Title: {row['title']}",
            # f"Author: {row['author']}",
            # f"Publisher: {row['publisher']}",
            f"Category: {row['category']}",
            f"Description: {row['description']}"
        ]
        return " ".join( # 리스트의 문자열들을 공백으로 연결할건데.....
            [p for p in parts if isinstance(p, str)] # NaN이나 None이 있으면 제외함
        ) # 최종적으로 하나의 문장 형태로 반환한다고 함!! "Title: ... Author: ... Publisher: ... Description: ..."

    def embed_batch(self, batch):
        """
        embed_batch 함수:
        책 텍스트 리스트를 받아 E5 임베딩을 batch 단위로 만들어 반환함
        얘는 대량으로 처리하기 좋은 방법이라서..... 뭐 정기적으로 대량임베딩하거나 할때 좋대
        나중에 e5를 다른걸로 바꾸거나 뭐.... 모델 운영하다보면 semantic search가 부정확해지거나 할때 쓰면 굿

        어차피 우린 제목, 저자, 출판사, 설명 정도만 있어서... SBERT나 OpenAI emb 같은거 굳이...
        """
        self.model.to(self.device).eval()
        batch_texts = [f"passage: {t}" for t in batch["text"]] # 각 텍스트 앞에 passage:를 붙힘!(e5 권장; 문맥 signal)

        inputs = self.tokenizer(
            batch_texts, return_tensors="pt", 
            truncation=True, # 길이 넘치면 자름
            padding=True,  # batch 단위로 다른 문장 길이에 맞게 padding
            max_length=256 # 최대 토큰 길이 제한
        ).to(self.device)

        with torch.no_grad():  # gradient 비활성화
            outputs = self.model(**inputs) # 딕셔너리를 언패킹(**)하여 모델에 전달
        
            # 원래는 아웃풋이 다 토큰단위인데.... mean해줘서 문장단위로 임베딩하게 된다는뎅
            emb = outputs.last_hidden_state.mean(dim=1)  # mean pooling
            emb = torch.nn.functional.normalize(emb, p=2, dim=1) # 정규화

        # pytorch 텐서는 기본적으로 연산 그래프를 추적해서 back-prop을 계산하나봐
        # 근데 .cpu().numpy()는 오직 gradient 추적 없는 순수 값(텐서)만 가능한 OP라서
        # .detach()를 통해서 그래프를 끊고 순수 값으로 탈바꿈 시킨대
        emb = emb.detach().cpu().numpy().tolist()
        return {"embedding": emb}

In [4]:
from pinecone import Pinecone, ServerlessSpec

class VectorDBService:
    def __init__(self, api_key: str, 
                 index_name: str = "books-index", 
                 dimension: int = 384):
        """
        이 클래스로 말하자면,,, pinecone을 편하게 쓰기 위한 wrapper임!!
        Pinecone 초기화 및 인덱스 연결
        """
        self.pc = Pinecone(api_key=api_key)
        self.index_name = index_name

        # 인덱스 없으면 생성
        if index_name not in self.pc.list_indexes().names():
            self.pc.create_index(
                name=index_name,
                dimension=dimension,
                metric="cosine",
                spec=ServerlessSpec(cloud="aws", region="us-east-1")
            )

        # 인덱스 연결
        self.client = self.pc.Index(index_name)

    def add_vectors(self, ids: list[str], embeddings: list[list[float]], metadata: list[dict]):
        """벡터와 메타데이터를 DB에 저장"""
        vectors = [(i, e, m) for i, e, m in zip(ids, embeddings, metadata)]
        self.client.upsert(vectors=vectors)

    def query(self, embedding: list[float], top_k: int = 5):
        """쿼리 벡터로 DB에서 유사 벡터 검색"""
        return self.client.query(vector=embedding, top_k=top_k, include_metadata=True)

In [5]:
aladin_service = AladinBookFetchService(ttb_key=ALADIN_KEY)
e5_service = E5Embedding()
pinecone_service = VectorDBService(api_key=PINECONE_KEY)

# 책 불러오기
bestsellers = aladin_service.fetch_books(query_type="Bestseller", max_results=100)

# 임베딩 하기
# 배치 텍스트와 book_id를 같은 순서로 저장하니 배치단위로 처리해도 매칭 안전함
# batch_texts = ["book A text", "book B text", "book C text"]
batch_texts = []
book_ids = []
for idx, book in enumerate(bestsellers, start=1):
    print(f"{idx}. {book['title']} - {book['author']} ({book['publisher']})")
    print(f"=> {book['category']}")
    print(f"=> {book['description'][:100]}... \n")
    text = e5_service.build_text(book)
    batch_texts.append(text)
    book_ids.append(book['isbn13']) # 고유 ID, DB에 key로 사용

embeddings = e5_service.embed_batch({"text": batch_texts})["embedding"]
for emb in embeddings:
    print(f"{len(emb)}: {emb[:5]}")

/home/0uk/.local/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


1. 어스탐 경의 임사전언 - 이영도 (지은이) (황금가지)
=> ['소설/시/희곡', '판타지/환상문학', '한국판타지/환상소설']
=> 한국 단행본 출판 수출 역사를 뒤바꾸며 전 세계 17개 언어권 30여 개 나라에서 인기리에 출간되고 있는 『눈물을 마시는 새』의 저자, 이영도 작가의 신작 장편소설 『어스탐 경의 ... 

2. 팬텀 버스터즈 5 - S코믹스 - 네오쇼코 (지은이), 김지혜 (옮긴이) (㈜소미미디어)
=> ['만화', '본격장르만화', '판타지', '액션 판타지']
=> 갑자기 나타난 수수께끼의 훈남 음양사, 카데노코지 보탄. 그의 목적은 시시쿠노 가문에 대한 선전 포고와 모가리를 복종시키는 것이었다! 보탄에 의해 대량의 악령을 받아들이게 된 모가... 

3. 체인소 맨 21 - 후지모토 타츠키 (지은이) (학산문화사(만화))
=> ['만화', '본격장르만화', '판타지', '액션 판타지']
=> ... 

4. 트렌드 코리아 2026 - 2026 대한민국 소비트렌드 전망 - 김난도, 전미영, 최지혜, 권정윤, 한다혜, 이혜원, 이수진, 서유현, 전다현, 이준영, 이향은, 김나은 (지은이) (미래의창)
=> ['경제경영', '트렌드/미래전망', '트렌드/미래전망 일반']
=> 세상은 작용과 반작용, 치열한 정반합(正反合)의 소용돌이가 거세게 휘몰아치고 있다. 방향을 잡기 어려울 정도로 속도가 빠르고 정신이 없다. 그렇다면 우리의 방향타는 어디에 있는가?... 

5. 내가 키운 S급들 1~2 + 굿즈 패키지 세트 - 비완 (지은이), 근서 (원작), seri (각색) (JAYPLEMEDIA)
=> ['만화', '인터넷 연재 만화']
=> &lt;사망한 한유현의 스킬과 능력치가 두 배로 전이됩니다.&gt; 그리고 힘과 함께 동생의 기억도 흘러들어 온다. 마지막 보답의 지속 시간이 끝나 간다. 시간이 되면 라우치타스의... 

6. 처음 만나는 양자의 세계 - 양자 역학부터 양자 컴퓨터 까지 - 채은미 (지은이) (북플레저)
=> ['과학',

In [6]:
ids = book_ids
embs = embeddings
meta = bestsellers

pinecone_service.add_vectors(ids, embs, meta)

In [33]:
# test_emb = embeddings[0]
# test_text = "Title: 드래곤의 모험 Category: 판타지 Description: 막 용이 날아다니고 마법사가 나와서 드래곤 잡는 이야기"
test_text = "query: 판타지 소설, 드래곤, 마법사, 모험, 전투, 용이 등장하는 이야기"

test_emb = e5_service.embed_batch({
    "text": [test_text] # 지금은 그 text 하나만 넣으니까 리스트로 감싸라고 하네
})["embedding"]

results = pinecone_service.query(test_emb, top_k = 5)['matches']
for result in results:
    print(result)

{'id': '9791171715374',
 'metadata': {'author': '유선경 (지은이)',
              'category': ['인문학', '교양 인문학'],
              'cover': 'https://image.aladin.co.kr/product/37542/42/coversum/k192032124_1.jpg',
              'description': '전작 《하루 한 장 나의 어휘력을 위한 필사 노트》가 ‘어휘력+필사’의 조합으로 독자의 '
                             '호응을 얻었다면, 이번 책은 ‘필사+표현력’으로 확장한다. 단순히 좋은 문장을 옮겨 '
                             '쓰는 데 그치지 않고 자신의 언어로 재구성하여 표현할 수 있게 돕는다.',
              'isbn13': '9791171715374',
              'publisher': '위즈덤하우스',
              'title': '하루 한 장 나의 표현력을 위한 필사 노트 - 뭉툭한 생각을 정교하게 다듬어주는 표현력 되찾기'},
 'score': 0.8784495,
 'values': []}
{'id': '9791162544327',
 'metadata': {'author': '멜 로빈스 (지은이), 윤효원 (옮긴이)',
              'category': ['자기계발', '성공', '성공학'],
              'cover': 'https://image.aladin.co.kr/product/36949/35/coversum/k532030618_2.jpg',
              'description': '중요하지도 않은 것들을 신경 쓰느라 나를 소모하는 싸움에서 벗어나, 진정 중요한 것?즉 '
                             '자기 자신, 자기 행복, 목표, 인생?에 집중하는 방법을 알려준다. 수백만 